# Treasury Edition: The Debt Trap Analysis

This analysis examines the U.S. fiscal situation through two critical metrics that reveal the structural nature of America's debt trap: **The Scale of Debt** and **The Interest Cost**.

## Core Analysis Framework

### Key Charts
- **Chart 1 - The Scale of Debt**: U.S. Federal Debt as Percentage of GDP showing explosive, non-linear trajectory
- **Chart 2 - The Interest Cost**: U.S. Federal Net Interest Outlays revealing the painful feedback loop

### The Debt Trap Thesis
The combination of massive debt accumulation and rising interest rates creates a self-reinforcing cycle where:
1. **Higher debt levels** require more Treasury issuance
2. **Rising interest costs** consume growing portions of tax revenue  
3. **Fiscal inflexibility** necessitates even more debt to cover interest payments
4. **Market supply pressure** becomes a permanent structural feature

### Data Sources
- **GFDEGDQ188S**: Federal Debt as Percent of GDP (FRED) - **REAL DATA** ✅
- **A091RC1Q027SBEA**: Federal Net Interest Outlays (FRED) - **REAL DATA** ✅
- **Analysis Period**: Multi-decade view to capture structural trends
- **Data Quality**: Professional-grade Federal Reserve Economic Data for industry analysis

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas_datareader.data as web
import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("📊 Treasury Edition: The Debt Trap Analysis")
print(f"Analysis date: {datetime.datetime.now().strftime('%Y-%m-%d')}")

In [ ]:
# --- Configuration ---
print("⚙️  Setting up U.S. Debt Trap analysis...")

# Analysis timeframe - Multi-decade view to capture structural trends
START_DATE = datetime.datetime(1990, 1, 1)  # 35-year analysis for debt trajectory
END_DATE = datetime.datetime.now()

# Debt Analysis Data Series (FRED)
DEBT_SERIES = {
    'GFDEGDQ188S': 'Federal Debt as Percent of GDP',
    'A091RC1Q027SBEA': 'Federal Net Interest Outlays'
}

print(f"📅 Analysis Period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print("📊 Focus: Federal Debt Trajectory and Interest Cost Analysis")
print("💾 Data sources: FRED API for U.S. Federal Debt metrics")
print("🎯 Debt Trap Analysis: Scale of Debt + Interest Cost Feedback Loop")

In [ ]:
# --- Federal Debt Data Fetching ---
print("🚀 Fetching REAL U.S. Federal Debt data from FRED...")

import time
from fredapi import FRED
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def fetch_fred_data_robust():
    """Fetch real FRED data with multiple fallback methods for professional analysis"""
    debt_data = {}
    
    # Method 1: Try FRED API directly (most reliable)
    try:
        print("   🔄 Method 1: Attempting FRED API connection...")
        fred = FRED(api_key=None)  # Will use default or environment variable
        
        for series, description in DEBT_SERIES.items():
            print(f"     • Fetching {description}...")
            try:
                data = fred.get_series(series, start=START_DATE, end=END_DATE)
                debt_data[series] = data
                print(f"       ✅ SUCCESS: Got {len(data)} observations")
                time.sleep(0.5)  # Rate limiting
            except Exception as e:
                print(f"       ❌ FRED API failed for {series}: {e}")
        
        if len(debt_data) == len(DEBT_SERIES):
            print("   ✅ FRED API: All series fetched successfully!")
            return debt_data
            
    except Exception as e:
        print(f"   ❌ FRED API connection failed: {e}")
    
    # Method 2: Try pandas_datareader with retry logic
    try:
        print("   🔄 Method 2: Attempting pandas_datareader with retry...")
        
        # Configure session with retries
        session = requests.Session()
        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        
        for series, description in DEBT_SERIES.items():
            if series not in debt_data:  # Only fetch if not already retrieved
                print(f"     • Fetching {description}...")
                try:
                    data = web.DataReader(series, 'fred', start=START_DATE, end=END_DATE)
                    debt_data[series] = data[series]
                    print(f"       ✅ SUCCESS: Got {len(data)} observations")
                    time.sleep(1)  # Rate limiting
                except Exception as e:
                    print(f"       ❌ Failed: {e}")
        
        if len(debt_data) == len(DEBT_SERIES):
            print("   ✅ pandas_datareader: All series fetched successfully!")
            return debt_data
            
    except Exception as e:
        print(f"   ❌ pandas_datareader failed: {e}")
    
    # Method 3: Direct FRED CSV download (last resort for real data)
    try:
        print("   🔄 Method 3: Attempting direct FRED CSV download...")
        
        fred_urls = {
            'GFDEGDQ188S': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GFDEGDQ188S',
            'A091RC1Q027SBEA': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=A091RC1Q027SBEA'
        }
        
        for series, description in DEBT_SERIES.items():
            if series not in debt_data and series in fred_urls:
                print(f"     • Downloading {description} CSV...")
                try:
                    response = requests.get(fred_urls[series], timeout=30)
                    response.raise_for_status()
                    
                    # Parse CSV data
                    from io import StringIO
                    csv_data = pd.read_csv(StringIO(response.text), index_col=0, parse_dates=True)
                    csv_data = csv_data.dropna()
                    
                    # Filter date range
                    mask = (csv_data.index >= START_DATE) & (csv_data.index <= END_DATE)
                    filtered_data = csv_data[mask].iloc[:, 0]  # First column
                    
                    debt_data[series] = filtered_data
                    print(f"       ✅ SUCCESS: Got {len(filtered_data)} observations")
                    time.sleep(1)
                    
                except Exception as e:
                    print(f"       ❌ CSV download failed: {e}")
        
        if len(debt_data) >= 1:  # At least one series
            print(f"   ⚠️  Partial success: Got {len(debt_data)}/{len(DEBT_SERIES)} series")
            return debt_data
            
    except Exception as e:
        print(f"   ❌ Direct CSV download failed: {e}")
    
    # If we reach here, all methods failed
    print("   ❌ ALL METHODS FAILED - Cannot proceed with simulated data for professional newsletter")
    raise Exception("Unable to fetch real FRED data. Professional newsletter requires authentic data.")

# Execute robust data fetching
try:
    debt_data = fetch_fred_data_robust()
    
    # Combine all debt data
    debt_df = pd.DataFrame(debt_data)
    
    # Forward fill missing data (quarterly to daily interpolation)
    debt_df = debt_df.ffill()
    
    # Clean data
    debt_df = debt_df.dropna()
    
    print(f"\n✅ REAL Federal debt data loaded successfully!")
    print(f"   • Data Source: Federal Reserve Economic Data (FRED)")
    print(f"   • Date range: {debt_df.index[0].strftime('%Y-%m-%d')} to {debt_df.index[-1].strftime('%Y-%m-%d')}")
    print(f"   • Available series: {len(debt_df.columns)}")
    print(f"   • Total observations: {len(debt_df)}")
    print(f"   • Data Quality: PROFESSIONAL GRADE ✅")
    
    # Current debt metrics snapshot
    print(f"\n📊 CURRENT DEBT METRICS (REAL DATA):")
    if 'GFDEGDQ188S' in debt_df.columns:
        current_debt_gdp = debt_df['GFDEGDQ188S'].iloc[-1]
        print(f"   • Federal Debt as % of GDP: {current_debt_gdp:.1f}%")
    
    if 'A091RC1Q027SBEA' in debt_df.columns:
        current_interest = debt_df['A091RC1Q027SBEA'].iloc[-1]
        print(f"   • Federal Net Interest Outlays: ${current_interest:.0f} billion")

except Exception as e:
    print(f"❌ CRITICAL ERROR: {e}")
    print("🚨 CANNOT PROCEED: Professional newsletter requires real FRED data")
    print("💡 Please check internet connection and FRED API access")
    raise Exception("Real data required for professional analysis")

In [ ]:
# --- CHART 1: THE SCALE OF DEBT ---
print("📊 Creating Chart 1: The Scale of Debt...")

# Import required plotting libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import seaborn as sns

# Set up professional chart styling
try:
    sns.set_style("whitegrid")
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    # Fallback to basic styling if seaborn styles not available
    plt.style.use('default')
    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.alpha'] = 0.3

# Create debt-to-GDP chart
fig, ax = plt.subplots(figsize=(14, 8))

if 'GFDEGDQ188S' in debt_df.columns:
    debt_gdp = debt_df['GFDEGDQ188S'].dropna()
    
    # Plot the debt trajectory with explosive visual emphasis
    ax.plot(debt_gdp.index, debt_gdp.values, linewidth=4, color='#d62728', alpha=0.9, label='Federal Debt/GDP')
    
    # Fill area under the curve to emphasize scale
    ax.fill_between(debt_gdp.index, 0, debt_gdp.values, alpha=0.3, color='#d62728')
    
    # Highlight key crisis periods that accelerated debt growth
    crisis_periods = [
        (datetime.datetime(2007, 1, 1), datetime.datetime(2009, 12, 31), '2008 Financial Crisis'),
        (datetime.datetime(2020, 1, 1), datetime.datetime(2021, 12, 31), 'COVID-19 Pandemic')
    ]
    
    for start, end, label in crisis_periods:
        ax.axvspan(start, end, alpha=0.2, color='orange', label=label if label == '2008 Financial Crisis' else '')
    
    # Add trend line to show acceleration
    years = [(date - debt_gdp.index[0]).days / 365.25 for date in debt_gdp.index]
    z = np.polyfit(years, debt_gdp.values, 2)  # Quadratic fit to show non-linear growth
    trend_line = np.poly1d(z)
    ax.plot(debt_gdp.index, trend_line(years), '--', linewidth=2, color='black', alpha=0.7, label='Non-linear Trend')

# Chart formatting
ax.set_title('The Scale of Debt', fontsize=18, fontweight='bold', pad=20)
ax.set_ylabel('Federal Debt (% of GDP)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=14, fontweight='bold')

# Professional date formatting
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_minor_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(DateFormatter('%Y'))

# Grid and styling
ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# Add current value annotation
if 'GFDEGDQ188S' in debt_df.columns:
    current_debt = debt_df['GFDEGDQ188S'].iloc[-1]
    ax.annotate(f'Current: {current_debt:.1f}%', 
                xy=(debt_df.index[-1], current_debt),
                xytext=(10, 10), textcoords='offset points',
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

# Analysis summary
if 'GFDEGDQ188S' in debt_df.columns:
    current_debt = debt_df['GFDEGDQ188S'].iloc[-1]
    debt_30_years_ago = debt_df['GFDEGDQ188S'].iloc[0] if len(debt_df) > 0 else None
    
    print(f"\n✅ DEBT SCALE ANALYSIS:")
    print(f"   • Current Federal Debt: {current_debt:.1f}% of GDP")
    if debt_30_years_ago:
        debt_increase = current_debt - debt_30_years_ago
        print(f"   • Increase since {debt_df.index[0].year}: +{debt_increase:.1f} percentage points")
    print(f"   • Trajectory: NON-LINEAR (Explosive Growth)")
    print(f"   • Market Implication: STRUCTURAL supply pressure, not temporary")

In [ ]:
# --- CHART 2: THE INTEREST COST ---
print("📊 Creating Chart 2: The Interest Cost...")

# Create interest cost chart
fig, ax = plt.subplots(figsize=(14, 8))

if 'A091RC1Q027SBEA' in debt_df.columns:
    interest_outlays = debt_df['A091RC1Q027SBEA'].dropna()
    
    # Plot the interest cost trajectory showing the painful feedback loop
    ax.plot(interest_outlays.index, interest_outlays.values, linewidth=4, color='#ff7f0e', alpha=0.9, label='Net Interest Outlays')
    
    # Fill area under the curve to emphasize the growing burden
    ax.fill_between(interest_outlays.index, 0, interest_outlays.values, alpha=0.3, color='#ff7f0e')
    
    # Set y-axis to start from 0 to show only positive values
    ax.set_ylim(bottom=0)
    
    # Highlight periods of accelerating interest costs
    # Find inflection points where interest costs accelerated
    recent_surge_start = datetime.datetime(2022, 1, 1)
    if recent_surge_start in interest_outlays.index or any(date >= recent_surge_start for date in interest_outlays.index):
        ax.axvspan(recent_surge_start, interest_outlays.index[-1], alpha=0.2, color='red', 
                   label='Rate Surge Era')
    
    # Add exponential trend line to show acceleration
    years = [(date - interest_outlays.index[0]).days / 365.25 for date in interest_outlays.index]
    # Use recent data for trend to show current trajectory
    recent_years = years[-20:] if len(years) > 20 else years
    recent_values = interest_outlays.values[-20:] if len(interest_outlays) > 20 else interest_outlays.values
    
    if len(recent_years) > 3:
        z = np.polyfit(recent_years, recent_values, 1)  # Linear fit for recent trend
        trend_line = np.poly1d(z)
        projected_years = years + [(years[-1] + i) for i in range(1, 6)]  # Project 5 years ahead
        full_trend = [trend_line(y) for y in projected_years]
        
        # Plot trend line
        trend_dates = list(interest_outlays.index) + [interest_outlays.index[-1] + datetime.timedelta(days=365*i) for i in range(1, 6)]
        ax.plot(trend_dates, full_trend, '--', linewidth=2, color='darkred', alpha=0.8, label='Current Trajectory')

# Chart formatting
ax.set_title('The Interest Cost', fontsize=18, fontweight='bold', pad=20)
ax.set_ylabel('Net Interest Outlays (Billions USD)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=14, fontweight='bold')

# Professional date formatting
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_minor_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(DateFormatter('%Y'))

# Grid and styling
ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# Add current value annotation
if 'A091RC1Q027SBEA' in debt_df.columns:
    current_interest = debt_df['A091RC1Q027SBEA'].iloc[-1]
    ax.annotate(f'Current: ${current_interest:.0f}B', 
                xy=(debt_df.index[-1], current_interest),
                xytext=(10, 10), textcoords='offset points',
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='orange', alpha=0.7))

plt.tight_layout()
plt.show()

# Interest cost analysis
if 'A091RC1Q027SBEA' in debt_df.columns:
    current_interest = debt_df['A091RC1Q027SBEA'].iloc[-1]
    interest_10_years_ago_idx = -40 if len(debt_df) > 40 else 0  # ~10 years ago (quarterly data)
    interest_10_years_ago = debt_df['A091RC1Q027SBEA'].iloc[interest_10_years_ago_idx]
    
    growth_rate = ((current_interest / interest_10_years_ago) ** (1/10) - 1) * 100 if interest_10_years_ago > 0 else 0
    
    print(f"\n✅ INTEREST COST ANALYSIS:")
    print(f"   • Current Net Interest Outlays: ${current_interest:.0f} billion")
    print(f"   • 10-Year Growth Rate: {growth_rate:.1f}% annually")
    print(f"   • Feedback Loop: ACTIVE (Higher rates = Higher costs)")
    print(f"   • Fiscal Impact: GROWING portion of tax revenue consumed")
    print(f"   • Debt Trap Status: ENGAGED (Interest costs drive more borrowing)")